# SEACO-6161 · Config and helpers · `dbw-arc-merone-pipeline-prod`

Shared setup for `01_investigation`, `01a_dbfs_scan`, `02_cleanup` and `03_weekly_vacuum_job`, which load it with
`%run ./00_config_and_helpers`. **You don't run this notebook on its own.**

| | |
|---|---|
| Workspace | `dbw-arc-merone-pipeline-prod` |
| DBFS root storage account | `dbstorage347sd45lirtsc` (workspace-managed, Standard_LRS, ADLS Gen2) |
| Size reported by Azure (May 2026) | 92.45 TiB |
| Cost (May 2026) | $1,046.39 / month |

**Cluster requirements:** use **Single User (dedicated)** access mode or No Isolation. Shared access mode blocks
`spark._jvm` and DBFS-root access, and these helpers need both.

In [0]:
# ---- Workspace-specific config -------------------------------------------------------------------------
WORKSPACE_NAME         = "dbw-arc-merone-pipeline-prod"
ROOT_STORAGE_ACCOUNT   = "dbstorage347sd45lirtsc"   # DBFS root account (bytes here = this cost line)
AZURE_REPORTED_TIB     = 92.45                        # from Azure metrics, May 2026
AZURE_MONTHLY_COST_USD = 1046.39                       # from Azure Cost Management, May 2026

# ---- Shared config ---------------------------------------------------------------------------------------
CATALOG            = ""                         # "" = auto-detect current catalog (hive_metastore / spark_catalog)
AUDIT_DB           = "storage_audit_seaco_6161" # small managed audit tables, written by every notebook
MIN_RETAIN_HOURS   = 168                        # never VACUUM below 7 days
RECENT_DAYS        = 30                         # a table written in the last N days counts as "active"
MAX_AUDIT_AGE_DAYS = 7                          # cleanup refuses to act on an inventory older than this
THREADS            = 16                         # driver threads for listing / DESCRIBE calls

In [0]:
import datetime as dt
import json
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import types as T

if not CATALOG:
    CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]

_jvm = spark._jvm
_hconf = spark._jsc.hadoopConfiguration()

TIB = 1024 ** 4
GIB = 1024 ** 3


def now_utc():
    return dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)


def human(n):
    if n is None:
        return "-"
    n = float(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB", "PiB"):
        if abs(n) < 1024 or unit == "PiB":
            return f"{n:,.2f} {unit}"
        n /= 1024


# ---- workspace identity ----------------------------------------------------------------------------------
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
API_URL = _ctx.apiUrl().get()
_API_TOKEN = _ctx.apiToken().get()
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl", "unknown")
RUN_USER = _ctx.userName().get() if _ctx.userName().isDefined() else "unknown"

print(f"Workspace URL : {WORKSPACE_URL}")
print(f"Config is for : {WORKSPACE_NAME}  (root storage {ROOT_STORAGE_ACCOUNT})")
print(f"Catalog       : {CATALOG}")
print(f"Audit db      : {AUDIT_DB}")
print(f"Run as        : {RUN_USER}")

In [0]:
# ---- Filesystem helpers (JVM side, fast) -----------------------------------------------------------------
def _fs_and_path(path):
    p = _jvm.org.apache.hadoop.fs.Path(path)
    return p.getFileSystem(_hconf), p


def content_summary(path):
    """Recursive size of a path, computed in the JVM. Returns dict(bytes, files, dirs, error)."""
    try:
        fs, p = _fs_and_path(path)
        cs = fs.getContentSummary(p)
        return dict(bytes=int(cs.getLength()), files=int(cs.getFileCount()),
                    dirs=int(cs.getDirectoryCount()), error=None)
    except Exception as e:  # missing path, permission, ...
        return dict(bytes=None, files=None, dirs=None, error=str(e)[:300])


def path_exists(path):
    try:
        fs, p = _fs_and_path(path)
        return bool(fs.exists(p))
    except Exception:
        return False


def newest_mtime(path):
    """Newest modification time (UTC datetime) of any file below path; None if empty."""
    fs, p = _fs_and_path(path)
    it = fs.listFiles(p, True)
    newest = 0
    while it.hasNext():
        newest = max(newest, it.next().getModificationTime())
    return dt.datetime.utcfromtimestamp(newest / 1000) if newest else None


def ls(path):
    try:
        return dbutils.fs.ls(path)
    except Exception as e:
        print(f"  ! cannot list {path}: {str(e)[:200]}")
        return []


def normalize(path):
    """dbfs:/a/b/ -> /a/b ; abfss URIs are lower-cased and kept as-is."""
    if path is None:
        return None
    p = path.strip()
    if "://" in p and not p.startswith("dbfs:"):
        return p.rstrip("/").lower()
    if p.startswith("dbfs:"):
        p = p[5:]
    if p.startswith("/dbfs/"):
        p = p[5:]
    return p.rstrip("/") or "/"


def is_under(path, parent):
    path, parent = normalize(path), normalize(parent)
    return path == parent or path.startswith(parent.rstrip("/") + "/")

In [0]:
# ---- Where do bytes live? ---------------------------------------------------------------------------------
# A path costs money on this line only if it is on ROOT_STORAGE_ACCOUNT. /mnt/* mount points can point
# anywhere, so every path is classified with the mount table (longest mount point wins).
_NON_ROOT_BUILTIN_SOURCES = {"databricks-datasets", "unitycatalogvolumes"}
MOUNTS = sorted(
    [(normalize(m.mountPoint), m.source) for m in dbutils.fs.mounts()],
    key=lambda x: -len(x[0]),
)
MOUNT_POINTS = [mp for mp, _ in MOUNTS if mp != "/"]


def _classify_source(source):
    s = (source or "").lower()
    if "://" in s:
        return "root" if ROOT_STORAGE_ACCOUNT in s else "external:" + s.split("/")[2]
    if s in _NON_ROOT_BUILTIN_SOURCES:
        return "external:" + s
    return "root"  # DatabricksRoot, databricks-results, mlflow-* ... live in the root account


def classify_path(path):
    """'root' | 'external:<account or source>' | 'unknown'"""
    if not path:
        return "unknown"
    low = path.lower()
    if "://" in low and not low.startswith("dbfs:"):
        return "root" if ROOT_STORAGE_ACCOUNT in low else "external:" + low.split("/")[2]
    p = normalize(path)
    if p.lower().startswith(("/volumes", "/volume")):
        return "external:uc-volume"
    for mp, src in MOUNTS:
        if mp != "/" and (p == mp or p.startswith(mp + "/")):
            return _classify_source(src)
    return "root"


def print_mounts():
    rows = [(mp, src, _classify_source(src)) for mp, src in sorted(MOUNTS)]
    display(spark.createDataFrame(rows, "mount_point string, source string, storage_class string"))

In [0]:
# ---- Catalog helpers ----------------------------------------------------------------------------------
def qname(db, table=None):
    parts = [CATALOG, db] + ([table] if table else [])
    return ".".join(f"`{p}`" for p in parts)


def list_databases():
    return sorted(r[0] for r in spark.sql(f"SHOW DATABASES IN `{CATALOG}`").collect())


def list_tables(db):
    try:
        rows = spark.sql(f"SHOW TABLES IN {qname(db)}").collect()
    except Exception as e:
        print(f"  ! SHOW TABLES {db}: {str(e)[:200]}")
        return []
    try:
        views = {r["viewName"] for r in spark.sql(f"SHOW VIEWS IN {qname(db)}").collect()}
    except Exception:
        views = set()
    return sorted(r["tableName"] for r in rows if not r["isTemporary"] and r["tableName"] not in views)


def table_extended(db, table):
    """Type (MANAGED/EXTERNAL), Provider and Location from DESCRIBE TABLE EXTENDED."""
    out = {}
    for r in spark.sql(f"DESCRIBE TABLE EXTENDED {qname(db, table)}").collect():
        if r["col_name"] in ("Type", "Provider", "Location", "Created Time"):
            out[r["col_name"]] = r["data_type"]
    return out


def describe_detail(db, table):
    r = spark.sql(f"DESCRIBE DETAIL {qname(db, table)}").collect()[0].asDict()
    return r


_MAINTENANCE_OPS = ("VACUUM", "OPTIMIZE", "SET TBLPROPERTIES", "UNSET TBLPROPERTIES", "ANALYZE",
                    "UPGRADE PROTOCOL", "ADD CONSTRAINT", "DROP CONSTRAINT", "CHANGE COLUMN",
                    "ADD COLUMNS", "RENAME COLUMN", "FSCK", "REORG")


def history_summary(db, table):
    """Summarise DESCRIBE HISTORY. History only reaches back logRetentionDuration (default 30 days)."""
    rows = (spark.sql(f"DESCRIBE HISTORY {qname(db, table)}")
            .select("version", "timestamp", "operation", "operationParameters", "job", "notebook", "userName")
            .collect())
    if not rows:
        return {}
    rows.sort(key=lambda r: r["version"])
    vac = [r["timestamp"] for r in rows if (r["operation"] or "").startswith("VACUUM")]
    writes = [r for r in rows if not (r["operation"] or "").startswith(_MAINTENANCE_OPS)]
    clones = [dict(source=(r["operationParameters"] or {}).get("source"),
                   shallow=(r["operationParameters"] or {}).get("isShallow"))
              for r in rows if r["operation"] == "CLONE"]
    lw = writes[-1] if writes else None
    job = (lw["job"].asDict() if lw is not None and lw["job"] is not None else {})
    nb = (lw["notebook"].asDict() if lw is not None and lw["notebook"] is not None else {})
    return dict(
        history_ops=len(rows),
        history_first_ts=rows[0]["timestamp"],
        history_last_ts=rows[-1]["timestamp"],
        last_operation=rows[-1]["operation"],
        distinct_ops=",".join(sorted({r["operation"] for r in rows})),
        write_ops=len(writes),
        last_vacuum_ts=max(vac) if vac else None,
        last_write_ts=lw["timestamp"] if lw is not None else None,
        last_writer_user=lw["userName"] if lw is not None else None,
        last_writer_job_id=str(job.get("jobId")) if job.get("jobId") else None,
        last_writer_job_name=job.get("jobName"),
        last_writer_notebook_id=str(nb.get("notebookId")) if nb.get("notebookId") else None,
        clone_history=json.dumps(clones) if clones else None,
    )

In [0]:
# ---- REST API (jobs / clusters), using this notebook's own token ---------------------------------------
def api_get(path, params=None, retries=5):
    for attempt in range(retries):
        r = requests.get(f"{API_URL}{path}", headers={"Authorization": f"Bearer {_API_TOKEN}"},
                         params=params or {}, timeout=60)
        if r.status_code == 429 and attempt < retries - 1:  # rate limited
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        return r.json()


def api_paginate(path, key, params=None):
    params = dict(params or {})
    while True:
        page = api_get(path, params)
        yield from page.get(key, [])
        token = page.get("next_page_token")
        if not page.get("has_more") or not token:
            return
        params["page_token"] = token


# ---- Audit tables --------------------------------------------------------------------------------------
spark.sql(f"CREATE DATABASE IF NOT EXISTS {qname(AUDIT_DB)} "
          f"COMMENT 'SEACO-6161 DBFS root storage audit (safe to drop when the ticket is closed)'")


def audit_table(name):
    return qname(AUDIT_DB, name)


def save_audit(name, rows, schema=None):
    """Append rows (list of dicts) to an audit table, tagged with RUN_ID / run_ts / workspace."""
    if not rows:
        print(f"  (no rows for {name})")
        return None
    base = dict(run_id=RUN_ID, run_ts=RUN_TS, workspace=WORKSPACE_NAME)
    merged = [{**base, **r} for r in rows]
    if schema:  # build tuples in schema order so dict key order never matters
        names = T._parse_datatype_string(schema).fieldNames()
        df = spark.createDataFrame([tuple(m.get(n) for n in names) for m in merged], schema=schema)
    else:
        df = spark.createDataFrame([Row(**m) for m in merged])
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(audit_table(name))
    print(f"  saved {len(rows):,} rows -> {AUDIT_DB}.{name}")
    return df


def latest(name):
    """Rows of the most recent run_id in an audit table."""
    t = audit_table(name)
    last = spark.sql(f"SELECT run_id, max(run_ts) ts FROM {t} GROUP BY run_id ORDER BY ts DESC LIMIT 1").collect()
    if not last:
        raise ValueError(f"{AUDIT_DB}.{name} is empty; run 01_investigation first")
    return spark.table(t).filter(F.col("run_id") == last[0]["run_id"]), last[0]["ts"]


def log_action(step, target, dry_run, status, bytes_before=None, bytes_after=None, duration_s=None, detail=None,
               files_before=None, files_after=None, rows_before=None, rows_after=None):
    row = dict(step=step, target=target, dry_run=bool(dry_run), status=status,
               bytes_before=bytes_before, bytes_after=bytes_after,
               files_before=files_before, files_after=files_after,
               rows_before=rows_before, rows_after=rows_after,
               duration_s=float(duration_s) if duration_s is not None else None,
               detail=(detail or "")[:2000], run_user=RUN_USER, action_ts=now_utc())
    schema = ("run_id string, run_ts timestamp, workspace string, step string, target string, dry_run boolean, "
              "status string, bytes_before long, bytes_after long, files_before long, files_after long, "
              "rows_before long, rows_after long, "
              "duration_s double, detail string, "
              "run_user string, action_ts timestamp")
    save_audit("cleanup_log", [row], schema=schema)


RUN_ID = str(uuid.uuid4())
RUN_TS = now_utc()
print(f"RUN_ID {RUN_ID}  @ {RUN_TS:%Y-%m-%d %H:%M:%S} UTC")

In [0]:
%scala
// Shallow-clone dependency check (runs in the JVM because inputFiles() returns very large arrays).
// Input : temp view _clone_check_input(table_name, table_location)
// Output: temp view _clone_check_output(table_name, table_location, referenced_dir, n_files, error)
// A table whose current files live OUTSIDE its own location is a shallow clone (or similar). The
// directory it points into belongs to the SOURCE table, and a VACUUM on that source would break it.
import spark.implicits._

def normalizeDbfs(p: String): String = {
  var s = Option(p).getOrElse("")
  if (s.startsWith("dbfs:")) s = s.substring(5)
  s.stripSuffix("/")
}

def findShallowCloneRefs(): Unit = {
  val input = spark.table("_clone_check_input").as[(String, String)].collect()
  val results = input.par.flatMap { case (name, location) =>
    try {
      val loc = normalizeDbfs(location)
      val outside = spark.read.format("delta").load(location).inputFiles
        .map(normalizeDbfs).filterNot(f => f.startsWith(loc + "/"))
      outside.groupBy(f => f.substring(0, math.max(f.lastIndexOf('/'), 0)))
        .map { case (dir, fs) => (name, location, dir, fs.length.toLong, "") }.toSeq
    } catch {
      case e: Throwable => Seq((name, location, null: String, 0L, String.valueOf(e.getMessage).take(300)))
    }
  }.seq.toSeq
  results.toDF("table_name", "table_location", "referenced_dir", "n_files", "error")
    .createOrReplaceTempView("_clone_check_output")
  println(s"clone check: ${input.length} tables checked, ${results.count(_._3 != null)} external file groups")
}

In [0]:
%scala
// Distributed sizing: getContentSummary runs on the EXECUTORS (one Spark task per group of paths), not on
// driver threads. Throughput grows with cluster cores, which is what makes a 100+ TiB DBFS scan finish.
import org.apache.hadoop.fs.Path
import org.apache.hadoop.conf.Configuration
import scala.collection.JavaConverters._
import org.apache.spark.sql.functions.{col, lit, to_timestamp}

type SizeRow = (String, Option[Long], Option[Long], Option[Long], String, Long)

def sizeOnExecutors(paths: Seq[String], slicesPerCore: Int = 4): Array[SizeRow] = {
  if (paths.isEmpty) return Array.empty[SizeRow]
  // The driver's Hadoop settings travel as plain strings inside the task closure, which Spark always
  // Java-serializes. Broadcasting a SerializableWritable(Configuration) instead fails on clusters that set
  // spark.serializer = KryoSerializer (Sedona): "KryoException: Error during Java deserialization".
  val confEntries: Map[String, String] =
    spark.sparkContext.hadoopConfiguration.iterator.asScala.map(e => e.getKey -> e.getValue).toMap
  val slices = math.max(1, math.min(paths.length, spark.sparkContext.defaultParallelism * slicesPerCore))
  spark.sparkContext.setJobDescription(s"SEACO-6161 sizing ${paths.length} paths")
  spark.sparkContext.parallelize(paths, slices).mapPartitions { it =>
    val hconf = new Configuration()
    confEntries.foreach { case (k, v) => hconf.set(k, v) }
    it.map { p =>
      val t0 = System.currentTimeMillis
      try {
        val hp = new Path(if (p.contains(":")) p else "dbfs:" + p)
        val cs = hp.getFileSystem(hconf).getContentSummary(hp)
        (p, Some(cs.getLength), Some(cs.getFileCount), Some(cs.getDirectoryCount), null: String,
         System.currentTimeMillis - t0)
      } catch {
        case e: Throwable => (p, None, None, None, String.valueOf(e.getMessage).take(300), System.currentTimeMillis - t0)
      }
    }
  }.collect()
}

// Size every path in inputView(path) and publish outputView(path, bytes, files, dirs, error, millis).
def sizePathsToView(inputView: String, outputView: String): Unit = {
  val paths = spark.table(inputView).select("path").as[String].collect().toSeq.distinct
  sizeOnExecutors(paths).toSeq.toDF("path", "bytes", "files", "dirs", "error", "millis")
    .createOrReplaceTempView(outputView)
}

// Size planView(path, storage_class) in batches; each batch is appended to auditTable at once, so a cancel
// or the time budget keeps everything already sized. Run id / ts / workspace come from spark conf.
def scanInBatches(planView: String, auditTable: String, batchSize: Int, budgetMinutes: Double): Unit = {
  val runId = spark.conf.get("seaco6161.run_id")
  val runTs = spark.conf.get("seaco6161.run_ts")
  val ws = spark.conf.get("seaco6161.workspace")
  val plan = spark.table(planView).select("path", "storage_class").as[(String, String)].collect()
  val cls = plan.toMap
  val batches = plan.map(_._1).grouped(batchSize).toArray
  val started = System.currentTimeMillis
  val deadline = started + (budgetMinutes * 60000).toLong
  var i = 0
  var done = 0
  while (i < batches.length && System.currentTimeMillis < deadline) {
    val res = sizeOnExecutors(batches(i))
    res.toSeq.map { case (p, b, f, d, e, ms) => (p, cls(p), "dir", b, f, d, e, ms) }
      .toDF("path", "storage_class", "kind", "bytes", "files", "dirs", "error", "millis")
      .select(lit(runId).as("run_id"), to_timestamp(lit(runTs)).as("run_ts"), lit(ws).as("workspace"), col("*"))
      .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(auditTable)
    done += res.length
    i += 1
    val gib = res.flatMap(_._2).sum / math.pow(1024, 3)
    println(f"  batch $i/${batches.length}: ${res.length} paths, $gib%,.1f GiB, " +
            f"${(System.currentTimeMillis - started) / 60000.0}%.1f min elapsed, $done/${plan.length} done")
  }
  spark.sparkContext.setJobDescription(null)
  if (done < plan.length)
    println(s"Time budget reached: ${plan.length - done} paths left. Re-run with resume_run_id = $runId")
}

In [0]:
def clone_sources(clone_rows, table_rows):
    """Map: source table full_name -> list of clone tables that reference its files."""
    by_loc = [(normalize(t["location"]), t["full_name"]) for t in table_rows if t.get("location")]
    out, unresolved = {}, []
    for c in clone_rows:
        if not c.get("referenced_dir"):
            continue
        d = normalize(c["referenced_dir"])
        owners = [name for loc, name in by_loc if d == loc or d.startswith(loc + "/")]
        if owners:
            for o in owners:
                out.setdefault(o, set()).add(c["table_name"])
        else:
            unresolved.append(c)
    return {k: sorted(v) for k, v in out.items()}, unresolved